# `Neo4jGraphRAGRetriever`: GraphRAG on an existing Neo4j graph with Haystack

This notebook uses the public, read-only Neo4j demo database (`companies`: news articles, chunks and organizations) to show:

1. The four search types: `vector`, `vector_cypher`, `hybrid`, `hybrid_cypher`
2. `query_params` in a `retrieval_query`
3. A RAG `Pipeline`
4. Saving and loading the pipeline as YAML
5. An `Agent` using the retriever as tools

The package itself depends only on `haystack-ai`, `neo4j-graphrag` and `neo4j`. The embedder is your choice. Here we install `sentence-transformers-haystack`, because `all-MiniLM-L6-v2` matches the demo DB's `news_sbert` index.

In [1]:
# From a clone of the repo (or: pip install git+https://github.com/neo4j-labs/neo4j-haystack)
%pip install -q -e .. sentence-transformers-haystack python-dotenv

Note: you may need to restart the kernel to use updated packages.


## Credentials

The demo DB credentials are public. The OpenAI key is read from `.env` (or asked for with `getpass`) and never printed.

In [2]:
import os
from getpass import getpass

from dotenv import find_dotenv, load_dotenv

os.environ.setdefault("NEO4J_USERNAME", "companies")
os.environ.setdefault("NEO4J_PASSWORD", "companies")

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

OPENAI_API_KEY set: True


## 1. Four search types, one query

In [3]:
from haystack_integrations.components.embedders.sentence_transformers import SentenceTransformersTextEmbedder

from haystack_integrations.components.retrievers.neo4j_graphrag import Neo4jGraphRAGRetriever

URL = "neo4j+s://demo.neo4jlabs.com"

RETRIEVAL_QUERY = """
WITH node AS chunk, score
MATCH (article:Article)-[:HAS_CHUNK]->(chunk)
OPTIONAL MATCH (article)-[:MENTIONS]->(org:Organization)
OPTIONAL MATCH (org)-[:HAS_COMPETITOR]->(rival:Organization)
RETURN chunk.text AS text, article.id AS article_id, article.title AS title,
       toString(article.date) AS date, article.sentiment AS sentiment,
       collect(DISTINCT org.name)[..5] AS companies,
       collect(DISTINCT rival.name)[..5] AS competitors, score
ORDER BY score DESC
"""


def make_retriever(search_type, **kwargs):
    params = dict(
        url=URL,
        database="companies",
        search_type=search_type,
        vector_index_name="news_sbert",
        text_embedder=SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2"),
        cypher_version="25",  # the demo DB defaults to Cypher 5; neo4j-graphrag emits Cypher 25
        top_k=3,
    )
    if search_type.startswith("hybrid"):
        params["fulltext_index_name"] = "news_fulltext"
    if search_type.endswith("_cypher"):
        params["retrieval_query"] = RETRIEVAL_QUERY
    params.update(kwargs)
    return Neo4jGraphRAGRetriever(**params)


retrievers = {st: make_retriever(st) for st in ["vector", "vector_cypher", "hybrid", "hybrid_cypher"]}

/usr/local/python/3.14.2/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
QUERY = "Which companies are working on electric vehicle batteries?"

for search_type, retriever in retrievers.items():
    docs = retriever.run(query=QUERY)["documents"]
    print(f"\n=== {search_type} ===")
    for doc in docs:
        print(f"score={doc.score:.3f} | {doc.content[:90]!r}")
        print("   meta:", doc.meta)

Batches: 100%|██████████| 1/1 [00:00<00:00, 51.52it/s]



=== vector ===
score=0.794 | 'SHERIDAN, WYOMING, UNITED STATES, April 15, 2022 /EINPresswire.com/ -- According to the la'
   meta: {'id': '113e19b7ad97cb2f20ed34f3962f891d630cf52d', 'labels': ['Chunk'], 'element_id': '4:14b688aa-0200-40b1-a396-d141d8c3e402:153074'}
score=0.792 | 'The roads of the Netherlands are going green at impressive speeds – and the charging indus'
   meta: {'id': '0c8381d617cc79a93523b1d62aa176399c15f3c5', 'labels': ['Chunk'], 'element_id': '4:14b688aa-0200-40b1-a396-d141d8c3e402:161090'}
score=0.789 | 'In this article, we are going to discuss the 5 most profitable car companies in the world.'
   meta: {'id': 'c50bfa1cf7867ab2bb4441077d56e0cd9d510b91', 'labels': ['Chunk'], 'element_id': '4:14b688aa-0200-40b1-a396-d141d8c3e402:148033'}


Batches: 100%|██████████| 1/1 [00:00<00:00, 78.87it/s]



=== vector_cypher ===
score=0.794 | 'SHERIDAN, WYOMING, UNITED STATES, April 15, 2022 /EINPresswire.com/ -- According to the la'
   meta: {'article_id': 'ART64838841355', 'title': 'Battery Management System Market Size | Growth Trends Report, 2027', 'date': '2022-04-15T08:08:00Z', 'sentiment': 0.47, 'companies': ['Bristol-Myers Squibb', 'Valence Technology'], 'competitors': ['BP Capital', 'A123 Systems']}
score=0.792 | 'The roads of the Netherlands are going green at impressive speeds – and the charging indus'
   meta: {'article_id': 'ART172768958970', 'title': 'Fully charged: green business Delta charges electric vehicles at work', 'date': '2022-12-18T23:00:00Z', 'sentiment': 0.722, 'companies': ['Delta Motorsport', 'Delta Electronics'], 'competitors': ['United Airlines Holdings', 'American Airlines Group', 'Foxconn', 'LG Energy Solution', 'Digi-Triumph Technology']}
score=0.789 | 'In this article, we are going to discuss the 5 most profitable car companies in the world.'
   meta: {'

Batches: 100%|██████████| 1/1 [00:00<00:00, 69.47it/s]



=== hybrid ===
score=1.000 | 'As with the Subaru Solterra, the first electric car from Suzuki could be developed with To'
   meta: {'id': '740c73e718f6fb74f061a921848b9ebad5b9c1ed', 'labels': ['Chunk'], 'element_id': '4:14b688aa-0200-40b1-a396-d141d8c3e402:214133'}
score=1.000 | 'SHERIDAN, WYOMING, UNITED STATES, April 15, 2022 /EINPresswire.com/ -- According to the la'
   meta: {'id': '113e19b7ad97cb2f20ed34f3962f891d630cf52d', 'labels': ['Chunk'], 'element_id': '4:14b688aa-0200-40b1-a396-d141d8c3e402:153074'}
score=0.998 | 'The roads of the Netherlands are going green at impressive speeds – and the charging indus'
   meta: {'id': '0c8381d617cc79a93523b1d62aa176399c15f3c5', 'labels': ['Chunk'], 'element_id': '4:14b688aa-0200-40b1-a396-d141d8c3e402:161090'}


Batches: 100%|██████████| 1/1 [00:00<00:00, 76.29it/s]



=== hybrid_cypher ===
score=1.000 | 'As with the Subaru Solterra, the first electric car from Suzuki could be developed with To'
   meta: {'article_id': 'ART236396092172', 'title': 'Suzuki’s first electric SUV could be based on a Toyota', 'date': '2022-08-12T13:30:00Z', 'sentiment': 0.614, 'companies': ['Subaru', 'Toyota Motor', 'Futuro'], 'competitors': []}
score=1.000 | 'SHERIDAN, WYOMING, UNITED STATES, April 15, 2022 /EINPresswire.com/ -- According to the la'
   meta: {'article_id': 'ART64838841355', 'title': 'Battery Management System Market Size | Growth Trends Report, 2027', 'date': '2022-04-15T08:08:00Z', 'sentiment': 0.47, 'companies': ['Bristol-Myers Squibb', 'Valence Technology'], 'competitors': ['BP Capital', 'A123 Systems']}
score=0.998 | 'The roads of the Netherlands are going green at impressive speeds – and the charging indus'
   meta: {'article_id': 'ART172768958970', 'title': 'Fully charged: green business Delta charges electric vehicles at work', 'date': '2022-12-18

`vector` and `hybrid` return the chunk's own properties (embeddings stripped) plus `labels` and `element_id`. The `*_cypher` types return whatever the `retrieval_query` returns: article title, date, mentioned companies and their competitors.

## 2. `query_params`: filter by date

`retrieval_query` is fixed configuration. Values used in it (`$since`) are passed per call with `query_params`.

In [5]:
SINCE_QUERY = """
WITH node AS chunk, score
MATCH (article:Article)-[:HAS_CHUNK]->(chunk)
WHERE article.date >= datetime($since)
RETURN chunk.text AS text, article.id AS article_id, article.title AS title,
       toString(article.date) AS date, score
ORDER BY score DESC
"""

recent = make_retriever("vector_cypher", retrieval_query=SINCE_QUERY, top_k=5)
for doc in recent.run(query=QUERY, query_params={"since": "2023-01-01"})["documents"]:
    print(doc.meta["date"][:10], "|", doc.meta["title"])
recent.close()

Batches: 100%|██████████| 1/1 [00:00<00:00, 75.29it/s]


2023-06-09 | 5 Most Profitable Car Companies in the World
2023-02-24 | United States Quality Assurance as a Service Market Insights, Forecast to 2029
2023-02-24 | United States Quality Assurance as a Service Market Insights, Forecast to 2029


## 3. RAG pipeline

retriever → `ChatPromptBuilder` → `OpenAIChatGenerator`. The prompt shows graph context from `meta` when it's there.

In [6]:
from haystack import Pipeline
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.dataclasses import ChatMessage

template = [
    ChatMessage.from_system(
        "Answer using only the context. Cite articles as [article_id] when an article_id is given."
    ),
    ChatMessage.from_user(
        """Context:
{% for doc in documents %}
---
{% if doc.meta.article_id %}[{{ doc.meta.article_id }}] {% endif %}{% if doc.meta.title %}{{ doc.meta.title }}{% endif %}{% if doc.meta.date %} ({{ doc.meta.date[:10] }}){% endif %}
{% if doc.meta.companies %}Companies: {{ doc.meta.companies | join(', ') }}
{% endif %}{% if doc.meta.competitors %}Competitors: {{ doc.meta.competitors | join(', ') }}
{% endif %}{{ doc.content }}
{% endfor %}

Question: {{ question }}"""
    ),
]

rag = Pipeline()
rag.add_component("retriever", make_retriever("hybrid_cypher", top_k=5))
rag.add_component("prompt_builder", ChatPromptBuilder(template=template, required_variables=["question", "documents"]))
rag.add_component("llm", OpenAIChatGenerator(model="gpt-4o-mini"))
rag.connect("retriever.documents", "prompt_builder.documents")
rag.connect("prompt_builder.prompt", "llm.messages")

question = "Which companies are working on EV batteries, and who are their competitors?"
result = rag.run({"retriever": {"query": question}, "prompt_builder": {"question": question}})
print(result["llm"]["replies"][0].text)

Batches: 100%|██████████| 1/1 [00:00<00:00, 71.25it/s]


Several companies are working on EV batteries, with notable players including:

1. **BYD** - A Chinese company that has invested heavily in battery technology and supplies batteries to competitors like Tesla and Ford.
2. **Sunwoda** - A Shenzhen-based unicorn that focuses on EV batteries; it has seen significant growth but is smaller compared to leaders like CATL and BYD.
3. **CATL** (Contemporary Amperex Technology Co., Limited) - Dominates the EV battery market alongside BYD.

Competitors in the EV market include:
- **Tesla**, which has its own battery production and technology.
- **Ford**, which is expanding its EV offerings and sourcing batteries from various suppliers, including BYD.
- **General Motors** (not directly mentioned in the context provided but known to be a competitor in the EV space).
- **Volkswagen**, which is significantly ramping up its EV production and related battery supply.

Overall, the competition among these companies is characterized by cutthroat dynamics, 

## 4. Save and load the pipeline

Credentials are stored as env-var names only. The component lives under `haystack_integrations`, so `Pipeline.loads()` works with Haystack's default allowlist.

In [7]:
import yaml

yaml_str = rag.dumps()
params = yaml.safe_load(yaml_str)["components"]["retriever"]["init_parameters"]
print("username:", params["username"])
print("password:", params["password"])
print("OpenAI key in YAML:", os.environ["OPENAI_API_KEY"] in yaml_str)

loaded = Pipeline.loads(yaml_str)
result = loaded.run({"retriever": {"query": question}, "prompt_builder": {"question": question}})
print("\nLoaded pipeline answer:\n", result["llm"]["replies"][0].text)

username: {'env_vars': ['NEO4J_USERNAME'], 'strict': True, 'type': 'env_var'}
password: {'env_vars': ['NEO4J_PASSWORD'], 'strict': True, 'type': 'env_var'}
OpenAI key in YAML: False


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.70it/s]



Loaded pipeline answer:
 Several companies are involved in the production of EV batteries, notable among them are:

1. **BYD** - A Chinese company that produces its own batteries and chips and sells batteries to other automakers, including Tesla and Ford. It has seen significant support from the Chinese government and has aggressively entered the global EV market with vehicles priced as low as $11,000.

2. **Sunwoda Electronic Co.** - A Shenzhen-based battery supplier for various automakers, including Volvo and Xpeng. They are preparing for an IPO and have experienced substantial growth in their EV battery business.

3. **CATL (Contemporary Amperex Technology Co., Limited)** - Currently the dominant player in the EV battery market, often cited as a competitor to BYD.

Competitors in the EV battery sector also include:

- **Tesla** - While Tesla produces EVs and batteries, it purchases batteries from other manufacturers like BYD and CATL.
- **Ford** - In addition to producing vehicles,

## 5. Agent with two retriever tools

`parameters=` exposes only `query` to the LLM. `retrieval_query` is never a tool argument, so the model can't run arbitrary Cypher.

In [8]:
from haystack.components.agents import Agent
from haystack.tools import ComponentTool

QUERY_ONLY = {
    "type": "object",
    "properties": {"query": {"type": "string", "description": "What to search for"}},
    "required": ["query"],
}

graph_tool = ComponentTool(
    component=retrievers["hybrid_cypher"],
    name="company_graph_search",
    description=(
        "Search news chunks with graph context: article id, title, date, sentiment, mentioned companies "
        "and their competitors. Use for specific, analytical questions about companies; cite article ids."
    ),
    parameters=QUERY_ONLY,
)
theme_tool = ComponentTool(
    component=retrievers["vector"],
    name="news_theme_search",
    description="Semantic search over news text. Use for broad themes and trends.",
    parameters=QUERY_ONLY,
)

agent = Agent(
    chat_generator=OpenAIChatGenerator(model="gpt-4o-mini"),
    tools=[graph_tool, theme_tool],
    system_prompt="You answer questions about companies in the news. Use the tools, and cite article ids when you have them.",
)
agent.warm_up()


def ask(q):
    messages = agent.run(messages=[ChatMessage.from_user(q)])["messages"]
    used = [tc.tool_name for m in messages for tc in (m.tool_calls or [])]
    print(f"Q: {q}\nTools used: {used}\nA: {messages[-1].text}\n")


ask("Who are the competitors of the companies mentioned in news about EV batteries? Cite articles.")
ask("What broad themes show up in the news about artificial intelligence?")

Batches: 100%|██████████| 1/1 [00:00<00:00, 85.72it/s]
[#AC7C]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('100.61.241.245', 7687)) (ResolvedIPv4Address(('100.61.241.245', 7687))): TimeoutError('The read operation timed out')
Unable to retrieve routing information
Transaction failed and will be retried in 1.0109219926110762s (Unable to retrieve routing information)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.59it/s]


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.28it/s]
[#C704]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('100.61.241.245', 7687)) (ResolvedIPv4Address(('100.61.241.245', 7687))): TimeoutError('The read operation timed out')
Unable to retrieve routing information
Transaction failed and will be retried in 0.8569762995639444s (Unable to retrieve routing information)
Failed to invoke Tool `company_graph_search` with parameters {'query': 'Nissa

Q: Who are the competitors of the companies mentioned in news about EV batteries? Cite articles.
Tools used: ['news_theme_search', 'company_graph_search', 'company_graph_search', 'company_graph_search', 'company_graph_search', 'company_graph_search', 'company_graph_search', 'company_graph_search', 'company_graph_search']
A: Here are some of the companies mentioned in recent news about electric vehicle (EV) batteries and their competitors:

### 1. **Nissan Motor**
- **Competitors:** 
  - **Honda**
  - **Toyota Motor**
  
**Articles:**
- "Nissan has an all new sedan in the 2019 Altima" ([Article ID: ART51146273349](https://www.example.com)) discusses Nissan's competition in the automotive sector with other manufacturers like Honda and Toyota. 
- In another context, Nissan is involved in EV developments, as mentioned in articles related to Nissan LEAF details ([Article ID: ART212743726415](https://www.example.com)).

### 2. **General Motors (GM)**
- **Competitors:**
  - **Ford Motor Compa

Batches: 100%|██████████| 1/1 [00:00<00:00, 72.52it/s]


Q: What broad themes show up in the news about artificial intelligence?
Tools used: ['news_theme_search']
A: Here are some broad themes currently appearing in news about artificial intelligence (AI):

1. **Advancements in Deep Learning**: AI is becoming increasingly sophisticated through deep learning techniques, which leverage neural networks inspired by human brain structures. Applications span from natural language processing to image recognition, where AI systems are surpassing human expert benchmarks in some tasks.

2. **AI in Business Transformation**: Various industries, especially healthcare and retail, are significantly investing in AI technology to enhance processes, products, and analysis. AI is expected to improve customer experiences and operational efficiency by automating routine tasks.

3. **Data Utilization and Big Data**: With an explosion in data generation, AI is seen as a critical tool to analyze and derive insights from this data efficiently. Businesses are adopti

## 6. Clean up

In [9]:
for r in retrievers.values():
    r.close()
rag.get_component("retriever").close()
loaded.get_component("retriever").close()